# ExaHyPE 2 Tutorials - Part 4: A 2D Euler Shock with a-posteriori Limiting

## A 2D Riemann Problem

We solve the compressible Euler equations for a 2D Riemann (shock-tube) problem: a corner of low-density, low-pressure gas sits in a box of dense, high-pressure gas. When the simulation starts, the dense gas rushes into the corner, launching **shock waves** and **contact discontinuities**.

A high-order ADER-DG scheme is accurate where the flow is smooth. Near these discontinuities it overshoots (Gibbs oscillations). The overshoots can drive the density or pressure negative and crash the run. The **a-posteriori limiter** keeps the scheme accurate and stable. After each step it checks every cell. It rolls back a *troubled* cell, one with a discrete-maximum-principle violation or a physically inadmissible state, and recomputes it with a Finite Volumes scheme. Every other cell keeps the high-order solution.

## Importing the API

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../api/"))
import exahype2

## Defining the Project

A `[-1, 1]^2` box. The conserved state is `(rho, rho*u, rho*v, E)`.

In [ ]:
my_project = exahype2.Project(
    namespace    = ["tutorials", "exahype2", "limiting"],
    project_name = "EulerShock",
    executable   = "EulerShock",
    dimensions   = 2,
    size         = [2.0, 2.0],
    offset       = [-1.0, -1.0],
    periodic_bc  = [False, False],   # Dirichlet boundaries keep the outer state
)

## The Solver and the Limiter

We build a high-order `ADERDGSolver` and, when `use_limiting` is on, switch its a-posteriori subcell limiter on with `add_subcell_limiter`. The limiter recomputes a cell on a Finite Volume subgrid of `2 * order + 1` cells per axis. It reuses the very same Euler terms as the high-order scheme.

The limiter recomputes a cell when either test rejects its candidate. The discrete maximum principle takes the observables written in `Limiter.cpp` and requires the candidate to stay inside the range that the cell and its neighbours had one step earlier, widened by the `dmp_*` parameters. The troubled-cell test `isPhysicallyAdmissible`, in the same file, demands a finite state with positive density and pressure.

The `use_limiting` flag is the heart of this tutorial - see *Why We Need the Limiter - Try It Yourself* below.

In [ ]:
# >>> Try setting this to False and re-running everything to see why we need the
# >>> limiter: the pure high-order scheme blows up on the shocks.
use_limiting = True

order    = 5
unknowns = {"rho": 1, "j": 2, "E": 1}

# The accurate but oscillation-prone high-order scheme.
aderdg_solver = exahype2.dg.ader.Solver(
    name                 = "ADERDGSolver",
    order                = order,
    unknowns             = unknowns,
    auxiliary_variables  = {},
    time_stepping        = exahype2.TimeStepping.Adaptive(0.5),
)
aderdg_solver.set_pde_terms(
    initial_condition   = True,
    boundary_conditions = True,
    flux                = True,
    max_eigenvalue      = True,
)

if use_limiting:
    # The fallback scheme a troubled cell is recomputed with, plus the two
    # tests that decide whether a cell is troubled. Both tests live in Limiter.cpp.
    aderdg_solver.set_pde_terms(
        dmp_observables        = True,
        physical_admissibility = True,
    )
    aderdg_solver.add_subcell_limiter(
        exahype2.fv.Godunov(),
        number_of_dmp_observables = 4,      # density, both velocities, pressure
        dmp_relaxation_parameter  = 1.0e-4, # absolute floor on the widening
        dmp_differences_scaling   = 1.0e-3, # widening relative to the local spread
    )

my_project.add_solver(aderdg_solver)

## Building the Project

We register the implementation files we actually use and then generate and compile. `ADERDGSolver.cpp` implements the Euler terms. The high-order scheme and the subcell scheme both evaluate them. `Limiter.cpp` implements the two tests that decide whether a cell is troubled. The solver declares them only while the limiter is on.

In [ ]:
my_project.output.makefile.add_cpp_file("ADERDGSolver.cpp")
if use_limiting:
    my_project.output.makefile.add_cpp_file("Limiter.cpp")
my_project.generate_and_build()

## Running the Simulation

In [ ]:
import os, subprocess

# Runtime parameters - edit these freely without a rebuild.
end_time  = 2.0     # -et : physical end time
snapshots = 20      # -ns : number of output snapshots (drives the animation)
level     = 3       # -l  : uniform mesh refinement level

# Run with 4 OpenMP threads (the notebook equivalent of: export OMP_NUM_THREADS=4)
os.environ["OMP_NUM_THREADS"] = "4"

subprocess.run(
    ["./EulerShock", "-et", str(end_time), "-ns", str(snapshots), "-l", str(level)],
    check=True,
)

## Visualising the Result

We plot the energy from the `ADERDGSolver` (use the field dropdown to switch to density, momentum or pressure). Drag the time slider partway through the run, where the curved shock front, the contact discontinuities and the central jet of the 2D Riemann problem are clearest. With the limiter on the solution stays physical (positive density and pressure) right through these discontinuities.

The limiter also writes a `LimiterStatus` field beside the solution. Pick it in the dropdown to see which cells ran on the Finite Volume subgrid at each snapshot. Expect a thin band tracking the shocks and the contact.

In [ ]:
import importlib, os, sys
sys.path.insert(0, os.path.abspath("../../"))
from postprocessing import render_solution
importlib.reload(render_solution)
render_solution.show_solution("ADERDGSolver", field="E")

## Why We Need the Limiter - Try It Yourself

Go back to *The Solver and the Limiter* cell, set

```python
use_limiting = False
```

and re-run every cell from there. Now only the high-order ADER-DG scheme runs, without the Finite Volumes fallback, and it oscillates uncontrollably at the shock fronts. Within the first couple of percent of the run those oscillations drive the density negative, the state then grows without bound, the reported maximum eigenvalue jumps by more than an order of magnitude, and the time step collapses with it.

You do not have to interrupt anything. Once the time step underflows, ExaHyPE trips a non-critical assertion on it, dumps the final state and shuts down within seconds, and the non-zero exit status makes the run cell raise a `CalledProcessError`. The dumped snapshot is worth plotting: the densities and energies in it are many orders of magnitude away from anything physical, and both signs appear where only positive values are meaningful.

That is the whole point of a-posteriori limiting: for problems with shocks, a high-order scheme on its own oscillates, and the limiter is what makes it usable.

## Experiment

Quick try without a rebuild: set `level` to 4 in the run cell for sharper - but slower - shocks, then re-run from there.

For the other experiments (tuning `dmp_relaxation_parameter` / `dmp_differences_scaling`, or moving the initial discontinuity via `ShockPosition`) see the documentation page, *Part 4 - A 2D Euler Shock with a-posteriori Limiting*. In **Part 5 - 3D Elastic Waves and a Taste of HPC** you will take ExaHyPE 2 to three dimensions and measure how the solver scales across parallel hardware.